# Module 3.3 — Neural Operators (Full Implementation)

Complete PyTorch implementation of DeepONet and Fourier Neural Operator.

## Requirements
```
pip install torch matplotlib numpy scipy
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Problem: Learning the Solution Operator

We learn the solution operator for the 1D Poisson equation:

$$-u''(x) = f(x), \quad x \in [0, 1], \quad u(0) = u(1) = 0$$

Given a forcing function $f$, the operator $\mathcal{G}$ maps $f \mapsto u$.

In [ ]:
def solve_poisson(f_values, x_grid):
    """Solve -u''=f with u(0)=u(1)=0 using finite differences."""
    n = len(x_grid) - 2  # Interior points
    dx = x_grid[1] - x_grid[0]
    
    # Tridiagonal system: -u_{i-1} + 2u_i - u_{i+1} = dx^2 * f_i
    A = np.zeros((3, n))
    A[0, 1:] = -1      # Upper diagonal
    A[1, :] = 2         # Main diagonal
    A[2, :-1] = -1      # Lower diagonal
    
    rhs = dx**2 * f_values[1:-1]
    u_interior = solve_banded((1, 1), A, rhs)
    
    u = np.zeros_like(x_grid)
    u[1:-1] = u_interior
    return u

# Generate training data
N_train = 1000
N_test = 200
n_points = 64
x_grid = np.linspace(0, 1, n_points)

def random_forcing(n_samples, n_points):
    """Generate random forcing functions as truncated Fourier series."""
    x = np.linspace(0, 1, n_points)
    f_all = np.zeros((n_samples, n_points))
    for i in range(n_samples):
        n_modes = np.random.randint(1, 6)
        for k in range(1, n_modes + 1):
            a = np.random.randn()
            f_all[i] += a * np.sin(k * np.pi * x)
    return f_all

# Generate data
f_train = random_forcing(N_train, n_points)
u_train = np.array([solve_poisson(f, x_grid) for f in f_train])

f_test = random_forcing(N_test, n_points)
u_test = np.array([solve_poisson(f, x_grid) for f in f_test])

print(f'Training data: {f_train.shape} -> {u_train.shape}')
print(f'Test data: {f_test.shape} -> {u_test.shape}')

## 2. DeepONet Architecture

DeepONet has two sub-networks:
- **Branch network**: processes the input function $f$ (evaluated at sensor locations)
- **Trunk network**: processes the query location $x$
- **Output**: $u(x) \approx \sum_{k=1}^{p} b_k(f) \cdot t_k(x)$

In [ ]:
class DeepONet(nn.Module):
    def __init__(self, branch_input_dim, trunk_input_dim, hidden_dim, p):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Linear(branch_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, p)
        )
        self.trunk = nn.Sequential(
            nn.Linear(trunk_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, p)
        )
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, f_sensors, x_query):
        # f_sensors: (batch, n_sensors)
        # x_query: (n_query, 1)
        b = self.branch(f_sensors)   # (batch, p)
        t = self.trunk(x_query)      # (n_query, p)
        # Output: (batch, n_query)
        out = torch.einsum('bp,qp->bq', b, t) + self.bias
        return out

p = 50  # Basis dimension
model = DeepONet(n_points, 1, 128, p).to(device)
print(f'DeepONet parameters: {sum(p.numel() for p in model.parameters())}')

## 3. Training

In [ ]:
# Convert to tensors
f_train_t = torch.tensor(f_train, dtype=torch.float32).to(device)
u_train_t = torch.tensor(u_train, dtype=torch.float32).to(device)
x_query = torch.tensor(x_grid, dtype=torch.float32).reshape(-1, 1).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)

n_epochs = 5000
batch_size = 64
losses = []

for epoch in range(n_epochs):
    idx = np.random.choice(N_train, batch_size, replace=False)
    f_batch = f_train_t[idx]
    u_batch = u_train_t[idx]
    
    optimizer.zero_grad()
    u_pred = model(f_batch, x_query)
    loss = ((u_pred - u_batch) ** 2).mean()
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}: Loss = {loss.item():.6e}')

plt.semilogy(losses)
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('DeepONet Training'); plt.grid(True, alpha=0.3)
plt.show()

## 4. Evaluation

In [ ]:
f_test_t = torch.tensor(f_test, dtype=torch.float32).to(device)
u_test_t = torch.tensor(u_test, dtype=torch.float32).to(device)

with torch.no_grad():
    u_pred_test = model(f_test_t, x_query).cpu().numpy()

# Relative L2 errors
errors = np.linalg.norm(u_pred_test - u_test, axis=1) / np.linalg.norm(u_test, axis=1)
print(f'Mean relative L2 error: {errors.mean():.4e}')
print(f'Max relative L2 error: {errors.max():.4e}')

# Plot examples
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i in range(3):
    idx = np.random.randint(N_test)
    axes[0, i].plot(x_grid, f_test[idx], 'k-')
    axes[0, i].set_title(f'Forcing f(x) #{idx}')
    axes[1, i].plot(x_grid, u_test[idx], 'b-', lw=2, label='True')
    axes[1, i].plot(x_grid, u_pred_test[idx], 'r--', lw=2, label='DeepONet')
    axes[1, i].legend()
    axes[1, i].set_title(f'Solution u(x), err={errors[idx]:.2e}')
plt.tight_layout()
plt.show()

## 5. Fourier Neural Operator (FNO)

The FNO performs spectral convolution in Fourier space:

$$(\mathcal{K}v)(x) = \mathcal{F}^{-1}\left(R \cdot \mathcal{F}(v)\right)(x)$$

In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes
        scale = 1 / (in_channels * out_channels)
        self.weights = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, x):
        # x shape: (batch, channels, spatial)
        x_ft = torch.fft.rfft(x)
        out_ft = torch.zeros(x.shape[0], self.out_channels, x.shape[2]//2 + 1,
                            dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes] = torch.einsum('bix,iox->box', 
                                                  x_ft[:, :, :self.modes], self.weights)
        return torch.fft.irfft(out_ft, n=x.shape[2])

class FNO1d(nn.Module):
    def __init__(self, modes, width, n_layers=4):
        super().__init__()
        self.fc_in = nn.Linear(2, width)  # input: (f(x), x)
        self.convs = nn.ModuleList([SpectralConv1d(width, width, modes) for _ in range(n_layers)])
        self.ws = nn.ModuleList([nn.Conv1d(width, width, 1) for _ in range(n_layers)])
        self.fc_out = nn.Sequential(nn.Linear(width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, x):
        # x: (batch, n_points, 2) where channels are [f(x_i), x_i]
        x = self.fc_in(x)  # (batch, n_points, width)
        x = x.permute(0, 2, 1)  # (batch, width, n_points)
        for conv, w in zip(self.convs, self.ws):
            x1 = conv(x)
            x2 = w(x)
            x = torch.nn.functional.gelu(x1 + x2)
        x = x.permute(0, 2, 1)  # (batch, n_points, width)
        x = self.fc_out(x)      # (batch, n_points, 1)
        return x.squeeze(-1)

fno = FNO1d(modes=16, width=32).to(device)
print(f'FNO parameters: {sum(p.numel() for p in fno.parameters())}')